# 1. Loading the data.

In [ ]:
# 1. Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Load the datasets (Using the paths you provided)
train_df = pd.read_csv('/content/TrainData.csv')
test_df = pd.read_csv('/content/test.csv')

# 3. Preview the Data
print("Training Data Sample (First 5 Rows):")
display(train_df.head())

Training Data Sample (First 5 Rows):


,id,area_type,availability,location,size,total_sqft,bath,balcony,price
0,0,type_I,Ready To Move,Banashankari 2 nd Stage,3 BHK,1030.0,2.0,2.0,77.25
1,1,type_I,Ready To Move,Balagere,2 BHK,1210.0,2.0,1.0,83.00
2,2,type_I,17-Oct,Banashankari Stage V,3 BHK,1540.0,3.0,2.0,48.51
3,3,type_I,Ready To Move,Thigalarapalya,3 BHK,1830.0,4.0,2.0,135.00
4,4,type_III,Ready To Move,arudi,3 Bedroom,NaN,2.0,0.0,80.00


# 2. Data Types Identification

In [ ]:
# IDENTIFY DATA TYPES
print("\n--- Data Information & Types ---")
print(train_df.info())


--- Data Information & Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            10000 non-null  int64  
 1   area_type     10000 non-null  object 
 2   availability  10000 non-null  object 
 3   location      10000 non-null  object 
 4   size          9987 non-null   object 
 5   total_sqft    9967 non-null   float64
 6   bath          9936 non-null   float64
 7   balcony       9525 non-null   float64
 8   price         10000 non-null  float64
dtypes: float64(4), int64(1), object(4)
memory usage: 703.3+ KB
None


# 3. Descriptive Statistics

In [ ]:
train_df.describe()

,id,total_sqft,bath,balcony,price
count,10000.00000,9967.000000,9936.000000,9525.000000,10000.000000
mean,4999.50000,1570.095822,2.692029,1.585302,113.275879
std,2886.89568,1302.566836,1.274172,0.814347,151.802643
min,0.00000,1.000000,1.000000,0.000000,8.000000
25%,2499.75000,1100.000000,2.000000,1.000000,50.000000
50%,4999.50000,1279.000000,2.000000,2.000000,72.000000
75%,7499.25000,1682.500000,3.000000,2.000000,120.000000
max,9999.00000,52272.000000,18.000000,3.000000,3600.000000


# 4.Handling Missing Values

In [ ]:
print("--- Missing Values in Training Data ---")
print(train_df.isnull().sum())

print("\n--- Missing Values in Test Data ---")
print(test_df.isnull().sum())

--- Missing Values in Training Data ---
id                0
area_type         0
availability      0
location          0
size             13
total_sqft       33
bath             64
balcony         475
price             0
dtype: int64

--- Missing Values in Test Data ---
id                0
area_type         0
availability      0
location          0
size              3
total_sqft       13
bath              9
balcony         134
dtype: int64


In [ ]:
# STEP 1: CLEANING DATA TYPES BEFORE IMPUTATION

# Function to handle 'total_sqft' (Converting ranges like '1100-1200' to average)
def convert_sqft_to_num(x):
    try:
        if isinstance(x, str):
            tokens = x.split('-')
            if len(tokens) == 2:
                return (float(tokens) + float(tokens[1])) / 2
            return float(x)
        return float(x)
    except:
        return None # Return None if the data is garbage

# Apply conversion so we can calculate Median later
train_df['total_sqft'] = train_df['total_sqft'].apply(convert_sqft_to_num)
test_df['total_sqft'] = test_df['total_sqft'].apply(convert_sqft_to_num)


# Transform the 'size' column (e.g., "2 BHK" -> 2.0)
# We take the first part of the string (e.g., "2") and convert it to a number.
train_df['size'] = train_df['size'].str.split(' ').str[0].astype(float)
test_df['size'] = test_df['size'].str.split(' ').str[0].astype(float)

# 2. Re-fill any NaNs this might have created
median_size = train_df['size'].median()
train_df['size'].fillna(median_size, inplace=True)
test_df['size'].fillna(median_size, inplace=True)

# ---------------------------------------------------------
# STEP 2: IMPUTATION
# ---------------------------------------------------------

# A. Numerical Imputation (Median)
# Note: We use TRAIN statistics to fill TEST data to avoid "Data Leakage"
median_sqft = train_df['total_sqft'].median()
median_bath = train_df['bath'].median()
median_balcony = train_df['balcony'].median()

# Fill Train Data
train_df['total_sqft'].fillna(median_sqft, inplace=True)
train_df['bath'].fillna(median_bath, inplace=True)
train_df['balcony'].fillna(median_balcony, inplace=True)


# Fill Test Data (using Train medians)
test_df['total_sqft'].fillna(median_sqft, inplace=True)
test_df['bath'].fillna(median_bath, inplace=True)
test_df['balcony'].fillna(median_balcony, inplace=True)

# B. Categorical Imputation (Mode)
# 'location' is the main categorical column we need
mode_location = train_df['location'].mode()

train_df['location'].fillna(mode_location, inplace=True)
test_df['location'].fillna(mode_location, inplace=True)
# ---------------------------------------------------------
# FINAL CHECK
# ---------------------------------------------------------
print("--- Missing Values in Train Data ---")
print(train_df.isnull().sum())
print("\n--- Missing Values in Test Data ---")
print(test_df.isnull().sum())



--- Missing Values in Train Data ---
id              0
area_type       0
availability    0
location        0
size            0
total_sqft      0
bath            0
balcony         0
price           0
dtype: int64

--- Missing Values in Test Data ---
id              0
area_type       0
availability    0
location        0
size            0
total_sqft      0
bath            0
balcony         0
dtype: int64


/tmp/ipython-input-2808496095.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['size'].fillna(median_size, inplace=True)
/tmp/ipython-input-2808496095.py:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

In [ ]:
train_df.head()

,id,area_type,availability,location,size,total_sqft,bath,balcony,price
0,0,type_I,Ready To Move,Banashankari 2 nd Stage,3.0,1030.0,2.0,2.0,77.25
1,1,type_I,Ready To Move,Balagere,2.0,1210.0,2.0,1.0,83.00
2,2,type_I,17-Oct,Banashankari Stage V,3.0,1540.0,3.0,2.0,48.51
3,3,type_I,Ready To Move,Thigalarapalya,3.0,1830.0,4.0,2.0,135.00
4,4,type_III,Ready To Move,arudi,3.0,1279.0,2.0,0.0,80.00


In [ ]:
test_df.head()

,id,area_type,availability,location,size,total_sqft,bath,balcony
0,0,type_II,Ready To Move,Banjara Layout,2.0,1050.0,2.0,1.0
1,1,type_I,Ready To Move,Rajiv Nagar,3.0,1690.0,3.0,1.0
2,2,type_II,Ready To Move,Hebbal,2.0,1100.0,2.0,1.0
3,3,type_III,Ready To Move,Munnekollal,6.0,1200.0,4.0,2.0
4,4,type_II,18-Apr,Choodasandra,4.0,2429.0,3.0,1.0


# 5. Outlier Handling

In [ ]:
print(f"Original Train Data Shape: {train_df.shape}")

# --- Rule 1: Handle Bathroom Outliers ---
# Logic: Bathrooms should not exceed Bedrooms + 2
# Note: Using 'bhk' if you renamed it, or 'size' if you didn't.
# Assuming 'size' holds the float value (e.g., 3.0) based on your last code.
train_df = train_df[train_df.bath <= (train_df['size'] + 2)]
print(f"Rows after removing Bathroom outliers: {train_df.shape}")

# --- Rule 2: Handle Sqft per Room Outliers (Engineering Logic) ---
# Logic: A room is at least 300 sqft.
train_df = train_df[~(train_df.total_sqft / train_df['size'] < 300)]
print(f"Rows after removing 'Small Bedroom' outliers: {train_df.shape}")

#---------Rule 3: Price Outlier Capping--------
# 1. Create helper column
train_df['price_per_sqft'] = train_df['price'] * 100000 / train_df['total_sqft']

def cap_pps_outliers(df):
    df_out = pd.DataFrame()

    # Group by location to compare apples-to-apples (e.g., Whitefield vs Kengeri)
    for key, subdf in df.groupby('location'):
        m = subdf.price_per_sqft.mean()
        st = subdf.price_per_sqft.std()

        # Define limits: We cap values that are outside 1 Standard Deviation
        # Logic: Pull extremes closer to the average without deleting them.
        lower_limit = m - st
        upper_limit = m + st

        # Apply Capping (Clip values to these limits)
        subdf_capped = subdf.copy()
        subdf_capped['price_per_sqft'] = subdf_capped['price_per_sqft'].clip(lower=lower_limit, upper=upper_limit)

        df_out = pd.concat([df_out, subdf_capped], ignore_index=True)

    return df_out

print(f"Original Train Data Shape: {train_df.shape}")

# 2. Apply Capping
train_df = cap_pps_outliers(train_df)

# 3. CRITICAL UPDATE: Recalculate 'price' based on the capped 'price_per_sqft'
# Since we adjusted the unit price, we must update the total price to match.
# Formula: Price (Lakhs) = (Price_Per_Sqft * Total_Sqft) / 100000
train_df['price'] = (train_df['price_per_sqft'] * train_df['total_sqft']) / 100000

print(f"Shape after Capping (Should match Original - No Data Loss): {train_df.shape}")

# 4. Drop the helper column
train_df = train_df.drop('price_per_sqft', axis=1)

Original Train Data Shape: (10000, 9)
Rows after removing Bathroom outliers: (9990, 9)
Rows after removing 'Small Bedroom' outliers: (9428, 9)
Original Train Data Shape: (9428, 10)
Shape after Capping (Should match Original - No Data Loss): (9428, 10)


In [ ]:
train_df.head()

,id,area_type,availability,location,size,total_sqft,bath,balcony,price
0,9662,type_II,Ready To Move,Anekal,1.0,351.0,1.0,1.0,16.0
1,5586,type_I,Ready To Move,Banaswadi,1.0,527.0,1.0,0.0,35.0
2,7411,type_I,Ready To Move,Basavangudi,1.0,670.0,1.0,1.0,50.0
3,457,type_I,Ready To Move,Devarabeesana Halli,2.0,1100.0,2.0,1.0,70.0
4,1199,type_II,Ready To Move,Devarabeesana Halli,3.0,1750.0,3.0,3.0,149.0


In [ ]:
test_df.head()

,id,area_type,availability,location,size,total_sqft,bath,balcony
0,0,type_II,Ready To Move,Banjara Layout,2.0,1050.0,2.0,1.0
1,1,type_I,Ready To Move,Rajiv Nagar,3.0,1690.0,3.0,1.0
2,2,type_II,Ready To Move,Hebbal,2.0,1100.0,2.0,1.0
3,3,type_III,Ready To Move,Munnekollal,6.0,1200.0,4.0,2.0
4,4,type_II,18-Apr,Choodasandra,4.0,2429.0,3.0,1.0


# 6. Scaling and Encoding

In [ ]:
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------
# 1. CLEANUP: Drop 'id' (Irrelevant Feature)
# ---------------------------------------------------------
# Source 4 (Image) shows an 'id' column. We drop it as it hurts the model.
if 'id' in train_df.columns:
    train_df.drop('id', axis=1, inplace=True)
if 'id' in test_df.columns:
    test_df.drop('id', axis=1, inplace=True)

# ---------------------------------------------------------
# 2. ENCODE 'availability' (In-Place Binary Encoding)
# ---------------------------------------------------------
# Logic: Convert 'Ready To Move' -> 1, Dates/Others -> 0 [Source 3]
def encode_availability(x):
    if x == 'Ready To Move':
        return 1
    return 0

train_df['availability'] = train_df['availability'].apply(encode_availability)
test_df['availability'] = test_df['availability'].apply(encode_availability)
print("Availability encoded to Binary (1 = Ready, 0 = Later).")

# ---------------------------------------------------------
# 3. ENCODE 'location' (Dimensionality Reduction)
# ---------------------------------------------------------
# Logic: Group rare locations (< 10 houses) into 'other' [Source 3]
# This reduces columns from ~1300 to ~240 without losing info.

# Step A: Strip extra spaces
train_df['location'] = train_df['location'].apply(lambda x: x.strip() if isinstance(x, str) else x)
test_df['location'] = test_df['location'].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Step B: Find rare locations (Use Train data only to avoid leakage)
location_stats = train_df['location'].value_counts()
locations_less_than_10 = location_stats[location_stats <= 10]

# Step C: Apply transformation
train_df['location'] = train_df['location'].apply(lambda x: 'other' if x in locations_less_than_10 else x)
test_df['location'] = test_df['location'].apply(lambda x: 'other' if x in locations_less_than_10 else x)

print(f"Unique Locations reduced to: {train_df['location'].nunique()}")

# ---------------------------------------------------------
# 4. SCALE NUMERICAL FEATURES (In-Place Standardization)
# ---------------------------------------------------------
# Logic: Scale inputs to Mean=0, Std=1.
# We scale 'total_sqft', 'bath', 'balcony', and 'size'. [Source 4]

scaler = StandardScaler()
cols_to_scale = ['size', 'total_sqft', 'bath', 'balcony']

# Handle potential missing values in these columns (fill with median)
# This prevents the Scaler from crashing.
for col in cols_to_scale:
    train_df[col] = train_df[col].fillna(train_df[col].median())
    test_df[col] = test_df[col].fillna(train_df[col].median()) # Use Train median for Test

# Fit on TRAIN, Transform BOTH
train_df[cols_to_scale] = scaler.fit_transform(train_df[cols_to_scale])
test_df[cols_to_scale] = scaler.transform(test_df[cols_to_scale])

print("\n--- Data Scaled In-Place (Z-Scores) ---")
display(train_df.head())
display(test_df.head())

Availability encoded to Binary (1 = Ready, 0 = Later).
Unique Locations reduced to: 178

--- Data Scaled In-Place (Z-Scores) ---


,area_type,availability,location,size,total_sqft,bath,balcony,price
0,type_II,1,Anekal,-1.691678,-0.947021,-1.464150,-0.763565,16.0
1,type_I,1,other,-1.691678,-0.814009,-1.464150,-2.022552,35.0
2,type_I,1,Basavangudi,-1.691678,-0.705938,-1.464150,-0.763565,50.0
3,type_I,1,other,-0.670802,-0.380966,-0.527777,-0.763565,70.0
4,type_II,1,other,0.350073,0.110270,0.408595,1.754409,149.0


,area_type,availability,location,size,total_sqft,bath,balcony
0,type_II,1,other,-0.670802,-0.418754,-0.527777,-0.763565
1,type_I,1,other,0.350073,0.064925,0.408595,-0.763565
2,type_II,1,Hebbal,-0.670802,-0.380966,-0.527777,-0.763565
3,type_III,1,Munnekollal,3.412701,-0.305391,1.344968,0.495422
4,type_II,0,Choodasandra,1.370949,0.623422,0.408595,-0.763565


In [ ]:
# ---------------------------------------------------------
# 1. ONE-HOT ENCODING (Aligning Train & Test)
# ---------------------------------------------------------

# A. Capture Target Variable
y = train_df['price']

# B. Combine Features temporarily
# We drop 'price' from Train to match Test's shape exactly
train_feats = train_df.drop('price', axis=1)
test_feats = test_df.copy()

# Concatenate (Stacking them)
all_data = pd.concat([train_feats, test_feats], axis=0)

# C. Apply One-Hot Encoding
# This creates columns like 'location_Whitefield', 'area_type_Super built-up Area'
all_data_encoded = pd.get_dummies(all_data, columns=['location', 'area_type'], drop_first=True)

# D. Split back into Train and Test (Submission)
# We use the original length of train_df to know where to cut
train_rows = len(train_df)

X = all_data_encoded.iloc[:train_rows, :]       # This is your "Train Data" for Step 2
X_submission = all_data_encoded.iloc[train_rows:, :]  # This is for Step 3 (later)

print(f"Features Ready.")
print(f"X Shape (Labeled Data): {X.shape}")
print(f"X_submission Shape (Unlabeled Data): {X_submission.shape}")

from sklearn.model_selection import train_test_split

# ---------------------------------------------------------
# 2. INTERNAL SPLIT (Train vs Validation)
# ---------------------------------------------------------
# We hold back 20% of the data (X_val, y_val) to test accuracy.
# The model will NEVER see this 20% during training.

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Set: {X_train.shape} houses")
print(f"Validation Set: {X_val.shape} houses")

Features Ready.
X Shape (Labeled Data): (9428, 318)
X_submission Shape (Unlabeled Data): (3320, 318)
Training Set: (7542, 318) houses
Validation Set: (1886, 318) houses


# 7. Model Training

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.metrics import r2_score, mean_squared_error


# Initialize the models
models = {
    "Linear Regression": LinearRegression(),
    "Lasso (L1)       ": Lasso(alpha=0.1), # Small alpha to not kill too many features
    "Random Forest    ": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "AdaBoost         ": AdaBoostRegressor(n_estimators=50, random_state=42)
}

print("--- MODEL PERFORMANCE REPORT ---")
print(f"{'Model Name':<20} | {'R2 Score':<10} | {'RMSE (Lakhs)':<10}")
print("-" * 45)

results = {}

for name, model in models.items():
    # A. Train
    model.fit(X_train, y_train)

    # B. Predict on Validation Data (Unseen by model)
    pred_val = model.predict(X_val)

    # C. Score
    r2 = r2_score(y_val, pred_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred_val))

    results[name] = r2

    print(f"{name:<20} | {r2:.4f}     | {rmse:.2f}")

print("-" * 45)

# Identify Top Model
best_model_name = max(results, key=results.get)
print(f"\nWinner: {best_model_name} (Score: {results[best_model_name]:.4f})")

--- MODEL PERFORMANCE REPORT ---
Model Name           | R2 Score   | RMSE (Lakhs)
---------------------------------------------
Linear Regression    | 0.6490     | 82.52
Lasso (L1)           | 0.6457     | 82.91
Random Forest        | 0.7533     | 69.18
Gradient Boosting    | 0.7560     | 68.81
AdaBoost             | 0.5591     | 92.49
---------------------------------------------

Winner: Gradient Boosting (Score: 0.7560)


# 8.Predicting on Test Data using the best model.

In [ ]:
# 1. Initialize the "Winner" Model
# We use the exact settings that gave R2=0.7560
final_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,  # Default
    max_depth=3,        # Default
    random_state=42
)

# 2. Train on the FULL dataset
print(f"Training Gradient Boosting on {X.shape} rows...")
final_model.fit(X, y)

# 3. Generate Predictions
# We predict on X_submission (the encoded test data from Source 1)
print(f"Generating predictions for {X_submission.shape} test houses...")
final_predictions = final_model.predict(X_submission)

# 4. Format Submission File
# Source 1 shows the 'id' column is a simple sequential index (0, 1, 2...)
submission_df = pd.DataFrame({
    'id': range(len(final_predictions)),
    'price': final_predictions
})

# 5. Export to CSV
file_name = 'submission_group4_GB.csv'
submission_df.to_csv(file_name, index=False)

print(f"\nSUCCESS! Submission file '{file_name}' is ready.")
print(f"Total Predictions: {len(submission_df)}")
display(submission_df.head())

Training Gradient Boosting on (9428, 318) rows...
Generating predictions for (3320, 318) test houses...

SUCCESS! Submission file 'submission_group4_GB.csv' is ready.
Total Predictions: 3320


,id,price
0,0,56.357085
1,1,113.455359
2,2,53.665981
3,3,124.066428
4,4,188.844815


# 9. Verification

In [ ]:
# ---------------------------------------------------------
# VERIFICATION: The "Twin House" Test (Fixed)
# ---------------------------------------------------------

verification_pairs = [
    # CASE 1: Small Apartment in Thippasandra
    # Test ID 20: 1 BHK, 650 sqft [Source 1] matches Train Row 52 (660 sqft, 30L) [Source 2]
    {'id': 20, 'expected_range': '30-40 Lakhs'},

    # CASE 2: Apartment in Hebbal
    # Test ID 2: 2 BHK, 1100 sqft [Source 1] matches Train Row 14 (1053 sqft, 48L) [Source 2]
    {'id': 2, 'expected_range': '45-55 Lakhs'},

    # CASE 3: Large Villa in Whitefield
    # Test ID 15: 4 Bedroom, 4356 sqft [Source 1] matches Train Row 17 (4400 sqft, 500L) [Source 2]
    {'id': 15, 'expected_range': '450-550 Lakhs'}
]

print("--- MODEL REALITY CHECK ---")
print(f"{'Test ID':<10} | {'Predicted Price':<20} | {'Expected (Based on Train Data)'}")
print("-" * 65)

for check in verification_pairs:
    # Get the predicted price array
    prices = submission_df.loc[submission_df['id'] == check['id'], 'price'].values

    # SAFETY FIX: Ensure we have a value and extract the first item
    if len(prices) > 0:
        pred_price = prices[0] # FIX: Extract the scalar from the array
        print(f"{check['id']:<10} | {pred_price:.2f} Lakhs{' '*8} | {check['expected_range']}")
    else:
        print(f"{check['id']:<10} | {'Not Found':<20} | {check['expected_range']}")

# ---------------------------------------------------------
# STATISTICAL CHECK
# ---------------------------------------------------------
avg_train = y.mean()
avg_pred = submission_df['price'].mean()

print("\n--- STATISTICAL CHECK ---")
print(f"Average Price in Training Data:   {avg_train:.2f} Lakhs")
print(f"Average Price in Your Prediction: {avg_pred:.2f} Lakhs")

--- MODEL REALITY CHECK ---
Test ID    | Predicted Price      | Expected (Based on Train Data)
-----------------------------------------------------------------
20         | 48.45 Lakhs         | 30-40 Lakhs
2          | 53.67 Lakhs         | 45-55 Lakhs
15         | 337.56 Lakhs         | 450-550 Lakhs

--- STATISTICAL CHECK ---
Average Price in Training Data:   107.24 Lakhs
Average Price in Your Prediction: 102.96 Lakhs


In [ ]:
pip install xgboost lightgbm catboost

In [ ]:
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Function to rename columns to generic 'f0', 'f1', ...
def rename_columns_for_lgbm(df):
    new_cols = [f'f{i}' for i in range(df.shape[1])]
    df.columns = new_cols
    return df

# Apply renaming to ensure unique and compatible feature names for LightGBM
X_train = rename_columns_for_lgbm(X_train.copy())
X_val = rename_columns_for_lgbm(X_val.copy())

# Convert DataFrames to NumPy arrays for models that expect them or don't handle feature names well
X_train_array = X_train.values
X_val_array = X_val.values

# ---------------------------------------------------------
# ADVANCED MODEL TRAINING
# ---------------------------------------------------------

# 1. XGBoost (Extreme Gradient Boosting)
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, n_jobs=-1, random_state=42)
xgb_model.fit(X_train_array, y_train)
xgb_pred = xgb_model.predict(X_val_array)

# 2. LightGBM (Light Gradient Boosting Machine)
print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05, n_jobs=-1, random_state=42)
lgb_model.fit(X_train, y_train) # Pass DataFrame directly to leverage column names
lgb_pred = lgb_model.predict(X_val) # Pass DataFrame directly to leverage column names for prediction

# 3. CatBoost (Categorical Boosting)
print("Training CatBoost...")
# verbose=0 silences the training output
cb_model = cb.CatBoostRegressor(n_estimators=1000, learning_rate=0.05, depth=6, verbose=0, random_state=42)
cb_model.fit(X_train_array, y_train)
cb_pred = cb_model.predict(X_val_array)

# ---------------------------------------------------------
# SCORING
# ---------------------------------------------------------
print("\n--- ADVANCED MODEL LEADERBOARD ---")
models = [('XGBoost', xgb_pred), ('LightGBM', lgb_pred), ('CatBoost', cb_pred)]

for name, pred in models:
    r2 = r2_score(y_val, pred)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    print(f"{name:10} | R2 Score: {r2:.4f} | RMSE: {rmse:.2f} Lakhs")


Training XGBoost...
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.049061 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 463
[LightGBM] [Info] Number of data points in the train set: 7542, number of used features: 94
[LightGBM] [Info] Start training from score 106.751078
Training CatBoost...

--- ADVANCED MODEL LEADERBOARD ---
XGBoost    | R2 Score: 0.8096 | RMSE: 60.77 Lakhs
LightGBM   | R2 Score: 0.7051 | RMSE: 75.64 Lakhs
CatBoost   | R2 Score: 0.8319 | RMSE: 57.11 Lakhs


In [ ]:
import catboost as cb
import pandas as pd

# 1. Convert X and X_submission DataFrames to NumPy arrays
X_array = X.values
X_submission_array = X_submission.values

# 2. Initialize CatBoostRegressor model with optimal parameters
final_cb_model = cb.CatBoostRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    depth=6,
    verbose=0,
    random_state=42
)

# 3. Train the CatBoostRegressor model on the full training dataset
print(f"Training CatBoost on {X_array.shape} rows...")
final_cb_model.fit(X_array, y)

# 4. Generate predictions on the X_submission_array
print(f"Generating predictions for {X_submission_array.shape} test houses...")
final_predictions_cb = final_cb_model.predict(X_submission_array)

# 5. Create a new pandas DataFrame named submission_df
submission_df = pd.DataFrame({
    'id': range(len(final_predictions_cb)),
    'price': final_predictions_cb
})

# 6. Export the submission_df to a CSV file
file_name_cb = 'submission_group4_CatBoost.csv'
submission_df.to_csv(file_name_cb, index=False)

print(f"\nSUCCESS! CatBoost submission file '{file_name_cb}' is ready.")
print(f"Total Predictions: {len(submission_df)}")
display(submission_df.head())

Training CatBoost on (9428, 318) rows...
Generating predictions for (3320, 318) test houses...

SUCCESS! CatBoost submission file 'submission_group4_CatBoost.csv' is ready.
Total Predictions: 3320


,id,price
0,0,59.362906
1,1,121.169758
2,2,62.197120
3,3,146.313398
4,4,165.725570


## Feature Importance Analysis

### Subtask:
Perform a feature importance analysis on the best-performing model (CatBoost) to understand which input features contribute most significantly to the price predictions.


**Reasoning**:
To understand which features are most influential, I will extract feature importances from the `final_cb_model` and associate them with their original feature names from the `X` DataFrame.



In [ ]:
import matplotlib.pyplot as plt

# 1. Extract feature importances from the trained CatBoost model
feature_importances = final_cb_model.get_feature_importance()

# 2. Create a DataFrame from the feature importances
# Ensure feature names are aligned with the original X DataFrame
# The current X has renamed columns (f0, f1, etc.) for LightGBM.
# We need to use the columns from the original X before this renaming if we want the actual names.
# However, since X_array was directly derived from the 'renamed' X, the features in the model correspond to f0, f1, etc.
# So, we'll map back to the 'f0, f1...' names for now.
feature_names = list(X.columns)
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

# 3. Sort the features by their importance in descending order
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Display the top 10 most important features
print("\n--- Top 10 Most Important Features (CatBoost) ---")
display(importance_df.head(10))


--- Top 10 Most Important Features (CatBoost) ---


,Feature,Importance
2,total_sqft,53.206277
3,bath,8.504366
314,location_other,7.668809
1,size,6.388617
4,balcony,5.718944
316,area_type_type_III,4.488265
0,availability,3.906488
89,location_Cunningham Road,1.971709
211,location_Malleshwaram,1.712857
253,location_Rajaji Nagar,1.708334
